In [ ]:
# -*- coding: utf-8 -*-
"""every_visit_monte_carlo_algorithm_01.ipynb

Automatically generated by Colab.


"""

# Monte Carlo (MC) de cada visita

"""

Buscador de tesoro en un laberinto
===================================


Dada una cuadricula que representa un laberinto el objetivo es estimar el valor
de cada casilla usando Monte Carlo de Cada Visita (Every-Visit MC) para un
agente en un laberinto 3x3, ahora con una política que le permite moverse en las
cuatro direcciones., basándose solo en la experiencia de varios intentos
(episodios).


+---+---+---+
| A | B | C |  <-- Fila 0
+---+---+---+
| D | E | F |  <-- Fila 1
+---+---+---+
| G | H | I |  <-- Fila 2
+---+---+---+
  ^   ^   ^
Col 0 Col 1 Col 2


Estados: {A, B, C, D, E, F, G, H, I}

Estado Inicial: Siempre empieza en A.

Estados Terminales: I (+10 recompensa al entrar), F (-5 recompensa al entrar).

Política (π) Fija: Desde cualquier casilla no terminal, el agente elige una de
                    las cuatro direcciones (Arriba, Abajo, Izquierda, Derecha)
                    con igual probabilidad (25% cada una). Si el movimiento
                    elegido choca contra un muro (fuera del 3x3), el agente se
                    queda en la misma casilla para ese paso.

Recompensas (R): +10 (entrar I), -5 (entrar F), -0.1 (cualquier otro movimiento
                  o quedarse quieto).

Factor de Descuento (γ): 1


----------------------------------
| A:       | B:       | C:       |
----------------------------------
| D:       | E:       | F (TRAP) |
----------------------------------
| G:       | H:       | I (TRSR) |
----------------------------------


"""

In [1]:
import random
import collections

# --- Environment Configuration ---
GRID_ROWS = 3
GRID_COLS = 3
START_STATE = (0, 0)  # Corresponds to 'A'
TREASURE_STATE = (2, 2)  # Corresponds to 'I'
TRAP_STATE = (1, 2)  # Corresponds to 'F'
TERMINAL_STATES = {TREASURE_STATE, TRAP_STATE}

# Rewards
REWARD_TREASURE = 10
REWARD_TRAP = -5
REWARD_STEP = -0.1  # Cost of moving

# Policy: Equiprobable random policy (25% chance for each direction)
ACTIONS = ["UP", "DOWN", "LEFT", "RIGHT"]

# Monte Carlo Parameters
GAMMA = 1.0  # Discount factor (1.0 for undiscounted returns)
NUM_EPISODES = 50000  # Number of episodes to simulate

# --- Helper Functions ---


def get_state_name(state_coord):
    """Maps (row, col) to letter names A-I for display."""
    row, col = state_coord
    return chr(ord("A") + row * GRID_COLS + col)


def is_valid_state(state):
    """Checks if a state is within the grid boundaries."""
    row, col = state
    return 0 <= row < GRID_ROWS and 0 <= col < GRID_COLS


def take_step(state, action):
    """
    Calculates the next state and reward based on the current state and action.
    Handles wall collisions.
    """
    if state in TERMINAL_STATES:
        return state, 0  # No movement or reward from terminal state

    row, col = state
    next_row, next_col = row, col

    if action == "UP":
        next_row -= 1
    elif action == "DOWN":
        next_row += 1
    elif action == "LEFT":
        next_col -= 1
    elif action == "RIGHT":
        next_col += 1

    next_state = (next_row, next_col)

    # Check for wall collision
    if not is_valid_state(next_state):
        next_state = state  # Stay in the same place if hit wall

    # Determine reward
    if next_state == TREASURE_STATE:
        reward = REWARD_TREASURE
    elif next_state == TRAP_STATE:
        reward = REWARD_TRAP
    else:
        # Reward is given for the *transition*, so apply step cost here
        reward = REWARD_STEP

    return next_state, reward


def generate_episode():
    """
    Generates a single episode following the equiprobable random policy.
    Returns a list of (state, reward) tuples, where reward is R_{t+1}
    received after being in state S_t.
    The list is [(S0, R1), (S1, R2), ..., (S_{T-1}, R_T)]
    """
    episode = []
    current_state = START_STATE
    steps = 0
    max_steps = GRID_ROWS * GRID_COLS * 5  # Avoid infinite loops in edge cases

    while current_state not in TERMINAL_STATES and steps < max_steps:
        action = random.choice(ACTIONS)
        next_state, reward = take_step(current_state, action)
        episode.append((current_state, reward))
        current_state = next_state
        steps += 1
    # Note: The final state (terminal) is not added to the episode list itself,
    # but the reward for reaching it IS the last reward in the list.
    if steps == max_steps:
        print("Warning: Episode reached max steps, might indicate an issue.")
    return episode


# --- Every-Visit Monte Carlo Algorithm ---

# Initialize V(s) for all states
V = collections.defaultdict(float)
# Keep track of sum of returns and visit counts for averaging
returns_sum = collections.defaultdict(float)
returns_count = collections.defaultdict(float)  # Now counts *every* visit

# Generate all possible states for initialization
all_states = [(r, c) for r in range(GRID_ROWS) for c in range(GRID_COLS)]
for s in all_states:
    V[s] = 0.0
    returns_sum[s] = 0.0
    returns_count[s] = 0.0

print(f"Running Every-Visit Monte Carlo for {NUM_EPISODES} episodes...")

for i in range(NUM_EPISODES):
    # 1. Generate an episode following the policy
    episode = generate_episode()
    # episode is like [(S0, R1), (S1, R2), ..., (S_{T-1}, R_T)]

    G = 0  # Return (cumulative discounted reward)
    # No need to track first visits within the episode for Every-Visit MC

    # 2. Loop backwards through the episode steps (t = T-1, T-2, ..., 0)
    for t in range(len(episode) - 1, -1, -1):
        state, reward = episode[t]  # S_t, R_{t+1}
        G = GAMMA * G + reward  # G is the return from state S_t onwards

        # 3. Update V(S_t) using the calculated return G for *every* visit
        #    No check for first visit is needed.
        returns_sum[state] += G
        returns_count[state] += 1
        # The value is the average of returns seen so far after *all* visits
        V[state] = returns_sum[state] / returns_count[state]

    # Optional: Print progress
    if (i + 1) % (NUM_EPISODES // 10) == 0:
        print(f"Episode {i + 1}/{NUM_EPISODES} completed.")

Running Every-Visit Monte Carlo for 50000 episodes...
Episode 5000/50000 completed.
Episode 10000/50000 completed.
Episode 15000/50000 completed.
Episode 20000/50000 completed.
Episode 25000/50000 completed.
Episode 30000/50000 completed.
Episode 35000/50000 completed.
Episode 40000/50000 completed.
Episode 45000/50000 completed.
Episode 50000/50000 completed.


In [2]:
# --- Output Results ---
print("\n--- Final State Values (Every-Visit MC) ---")  # Updated title

# Prepare for formatted output
output_grid = [["" for _ in range(GRID_COLS)] for _ in range(GRID_ROWS)]
max_len = 0

for r in range(GRID_ROWS):
    for c in range(GRID_COLS):
        state = (r, c)
        state_name = get_state_name(state)
        value_str = ""
        if state == TREASURE_STATE:
            value_str = f"{state_name}(TRSR): Goal"
        elif state == TRAP_STATE:
            value_str = f"{state_name}(TRAP): Goal"
        else:
            # Use the calculated value V[state]
            value_str = f"{state_name}: {V[state]:.2f}"

        output_grid[r][c] = value_str
        max_len = max(max_len, len(value_str))

# Print the grid
print("Estimated values V(s):")
for r in range(GRID_ROWS):
    row_str = "| "
    for c in range(GRID_COLS):
        row_str += output_grid[r][c].ljust(max_len) + " | "
    print("-" * len(row_str))
    print(row_str)
print("-" * len(row_str))

# Example of returns list for a state (optional check)
# print("\nExample Returns Count for state A (0,0):", returns_count[(0,0)])
# print("Example Returns Sum for state A (0,0):", returns_sum[(0,0)])


--- Final State Values (Every-Visit MC) ---
Estimated values V(s):
--------------------------------------------------
| A: -1.92      | B: -2.48      | C: -3.87      | 
--------------------------------------------------
| D: -0.92      | E: -1.34      | F(TRAP): Goal | 
--------------------------------------------------
| G: 0.83       | H: 3.05       | I(TRSR): Goal | 
--------------------------------------------------
